# Transcriber — Colab notebook

Transcribes long audio files with speaker diarization using WhisperX (faster-whisper + wav2vec2 alignment + pyannote 3.1). See the [project README](https://github.com/OWNER/REPO) for context.

**Before running:**
1. Accept pyannote model terms on both pages while logged in to Hugging Face:
   - https://huggingface.co/pyannote/speaker-diarization-3.1
   - https://huggingface.co/pyannote/segmentation-3.0
2. Create a read token at https://huggingface.co/settings/tokens.
3. In Colab, click the 🔑 icon in the sidebar, add a secret named `HF_TOKEN`, and toggle notebook access on.
4. **Runtime → Change runtime type → T4 GPU**.
5. Put your audio file on Google Drive, then update `AUDIO_FILE` in the config cell below.

## 1. Install dependencies

In [ ]:
!pip install -q whisperx

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configure

Token is read from Colab Secrets — never hardcode it into this notebook.

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

AUDIO_FILE = '/content/drive/MyDrive/transcripts/input.mp3'  # change me
OUTPUT_DIR = '/content/drive/MyDrive/transcripts'

LANGUAGE = 'en'
MODEL = 'large-v3'
COMPUTE_TYPE = 'float16'   # T4 has 16 GB VRAM — plenty of headroom
BATCH_SIZE = 16
MIN_SPEAKERS = 2
MAX_SPEAKERS = 10

## 4. Run the pipeline

Each stage loads its model, runs, and explicitly frees memory before the next stage loads. On a 6-hour file expect roughly 20–30 minutes total on a free T4.

In [ ]:
import gc
import torch
import whisperx

device = 'cuda' if torch.cuda.is_available() else 'cpu'
audio = whisperx.load_audio(AUDIO_FILE)

print('Transcribing...')
model = whisperx.load_model(MODEL, device, compute_type=COMPUTE_TYPE)
result = model.transcribe(audio, batch_size=BATCH_SIZE, language=LANGUAGE)
del model; gc.collect(); torch.cuda.empty_cache()

print('Aligning...')
align_model, metadata = whisperx.load_align_model(language_code=result['language'], device=device)
result = whisperx.align(result['segments'], align_model, metadata, audio, device)
del align_model; gc.collect(); torch.cuda.empty_cache()

print('Diarizing...')
diarize_model = whisperx.DiarizationPipeline(use_auth_token=HF_TOKEN, device=device)
diarize_segments = diarize_model(audio, min_speakers=MIN_SPEAKERS, max_speakers=MAX_SPEAKERS)
result = whisperx.assign_word_speakers(diarize_segments, result)
del diarize_model; gc.collect(); torch.cuda.empty_cache()

print('Done.')

## 5. Save outputs

Writes JSON (full, word-level), TXT (human-readable with speaker turns), and SRT (subtitles) to your Drive.

In [ ]:
import json
import os
from pathlib import Path

os.makedirs(OUTPUT_DIR, exist_ok=True)
stem = Path(AUDIO_FILE).stem
base = Path(OUTPUT_DIR) / stem

# Full JSON
with open(f'{base}.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

# Human-readable TXT grouped by speaker turn
def fmt_txt(result):
    lines, current, buf, start = [], None, [], 0.0
    for seg in result.get('segments', []):
        spk = seg.get('speaker', 'UNKNOWN')
        txt = seg.get('text', '').strip()
        if not txt:
            continue
        if spk != current:
            if buf:
                lines.append(f"[{start:.1f}s] {current}: {' '.join(buf).strip()}")
            buf, current, start = [], spk, float(seg.get('start', 0.0))
        buf.append(txt)
    if buf:
        lines.append(f"[{start:.1f}s] {current}: {' '.join(buf).strip()}")
    return '\n'.join(lines) + '\n'

with open(f'{base}.txt', 'w', encoding='utf-8') as f:
    f.write(fmt_txt(result))

# SRT subtitles
def fmt_ts(s):
    ms = int(round(s * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    sec, ms = divmod(ms, 1000)
    return f'{h:02d}:{m:02d}:{sec:02d},{ms:03d}'

with open(f'{base}.srt', 'w', encoding='utf-8') as f:
    for i, seg in enumerate(result.get('segments', []), start=1):
        f.write(f"{i}\n{fmt_ts(float(seg.get('start', 0.0)))} --> {fmt_ts(float(seg.get('end', 0.0)))}\n")
        f.write(f"{seg.get('speaker', 'UNKNOWN')}: {seg.get('text', '').strip()}\n\n")

print(f'Wrote {base}.json, {base}.txt, {base}.srt')